# AI512 Graded Lab — 4 Algorithms (Beginner Edition)
**One file · four parts · short explanations · commented, runnable Python.** Based on Lab 3: graph A*, 8-Queens SA/GA; Local Beam Search is an additional syllabus topic (different from the notebook’s level-based Beam Search).

## Part 1 — Local Beam Search
**Idea:** Keep `k` solutions. Generate all their neighbours, then retain the best `k`. Repeat. Here, higher score is better.

In [ ]:
def local_beam_search(start_states, neighbors, score, k=3, steps=20):
    states = start_states[:k]            # Current k solutions
    best = max(states, key=score)        # Best ever seen

    for _ in range(steps):
        candidates = []
        for state in states:             # Explore every current solution
            candidates += neighbors(state)
        if not candidates:               # No further moves
            break
        candidates = list(set(candidates))  # Avoid repeated states (hashable states)
        candidates.sort(key=score, reverse=True)  # Highest score first
        states = candidates[:k]          # Keep only the best k
        if score(states[0]) > score(best):
            best = states[0]
    return best

# Demo: maximize x, moving +1 or -1 within [0, 10]
def number_neighbors(x):
    return [y for y in (x - 1, x + 1) if 0 <= y <= 10]

print('Local Beam:', local_beam_search([1, 3, 5], number_neighbors, lambda x: x, 3, 10))

## Part 2 — Simulated Annealing (8-Queens)
**Idea:** Accept better moves; sometimes accept worse moves to escape a trap. Reduce temperature gradually. **Goal:** zero attacking pairs.

In [ ]:
import random
import math

N = 8

def conflicts(state):
    total = 0
    for c1 in range(N):
        for c2 in range(c1 + 1, N):
            same_row = state[c1] == state[c2]
            same_diagonal = abs(state[c1] - state[c2]) == abs(c1 - c2)
            if same_row or same_diagonal:
                total += 1
    return total

def change_one_queen(state):
    new_state = state.copy()            # Do not change original
    column = random.randrange(N)       # Pick a random column
    choices = [r for r in range(N) if r != state[column]]
    new_state[column] = random.choice(choices)  # Move queen to new row
    return new_state

def simulated_annealing(steps=20000, temperature=10.0, cooling=0.999):
    current = [random.randrange(N) for _ in range(N)]
    best = current.copy()
    for _ in range(steps):
        if conflicts(best) == 0:
            break
        candidate = change_one_queen(current)
        difference = conflicts(candidate) - conflicts(current)  # Lower is better
        if difference <= 0 or random.random() < math.exp(-difference / temperature):
            current = candidate         # Accept better or probabilistically worse move
        if conflicts(current) < conflicts(best):
            best = current.copy()       # Save best seen
        temperature = max(0.0001, temperature * cooling)
    return best, conflicts(best)

random.seed(7)
print('Simulated Annealing:', simulated_annealing())

## Part 3 — Genetic Algorithm (8-Queens)
**Idea:** Make a population → select strong parents → crossover → mutation → repeat. **Fitness:** `28 - conflicts`, so 28 is perfect.

In [ ]:
def fitness(state):
    return 28 - conflicts(state)         # 8 queens have 28 possible pairs

def genetic_algorithm(pop_size=100, generations=1000, mutation_rate=0.15):
    population = [[random.randrange(N) for _ in range(N)] for _ in range(pop_size)]

    def select_parent():
        group = random.sample(population, 3)  # Tournament selection
        return max(group, key=fitness)

    for _ in range(generations):
        population.sort(key=fitness, reverse=True)
        if fitness(population[0]) == 28:
            break
        next_population = [population[0].copy()]  # Elitism: preserve best
        while len(next_population) < pop_size:
            parent1 = select_parent()
            parent2 = select_parent()
            cut = random.randint(1, N - 1)
            child = parent1[:cut] + parent2[cut:]  # Crossover
            for column in range(N):
                if random.random() < mutation_rate:
                    child[column] = random.randrange(N)  # Mutation
            next_population.append(child)
        population = next_population
    best = max(population, key=fitness)
    return best, conflicts(best)

random.seed(7)
print('Genetic Algorithm:', genetic_algorithm())

## Part 4 — A* Search (Lab 3 graph)
**Idea:** Always expand the smallest `f = g + h`. `g` is actual cost so far; `h` is estimated cost remaining. Use a priority queue.

In [ ]:
import heapq

GRAPH = {
    'S': {'A': 2, 'B': 4, 'C': 3},
    'A': {'D': 4, 'E': 7},
    'B': {'E': 1, 'F': 5},
    'C': {'F': 2},
    'D': {'G': 5},
    'E': {'G': 3},
    'F': {'G': 2},
    'G': {}
}
# Lab 3 asks for H1. Use H1 values from the source notebook.
H1 = {'S': 6, 'A': 7, 'B': 4, 'C': 4, 'D': 5, 'E': 3, 'F': 2, 'G': 0}

def astar(graph, h, start='S', goal='G'):
    queue = [(h[start], 0, start)]       # (f, g, node)
    best_cost = {start: 0}              # Cheapest known g for each node
    parent = {start: None}              # Used to rebuild the path

    while queue:
        f, g, node = heapq.heappop(queue)  # Take smallest f
        if g != best_cost[node]:        # Ignore outdated queue entries
            continue
        if node == goal:
            path = []
            while node is not None:
                path.append(node)
                node = parent[node]
            return path[::-1], g        # Reverse path, return cost

        for neighbor, edge_cost in graph[node].items():
            new_g = g + edge_cost
            if new_g < best_cost.get(neighbor, float('inf')):
                best_cost[neighbor] = new_g
                parent[neighbor] = node
                heapq.heappush(queue, (new_g + h[neighbor], new_g, neighbor))
    return None, float('inf')

print('A*:', astar(GRAPH, H1))

### 20-second revision
- **Local Beam:** Generate neighbours → keep best `k`.
- **SA:** Random neighbour → accept based on temperature → cool.
- **GA:** Fitness → select → crossover → mutate.
- **A*:** Expand lowest `g+h` → update costs and parents.

**Practice:** Change `k`, cooling rate, mutation rate, and heuristic values; predict how behaviour changes.